# Tier 2 · Notebook 12 — Tokenization & BPE From Scratch

> **Goal:** build **byte-pair encoding** (BPE) — the tokenizer GPT actually uses — from nothing, understand the vocabulary/merge machinery, and see *why tokenization is the source of many of an LLM's weirdest failures* (bad arithmetic, can't-count-letters, non-English inefficiency).

**Why tokenization exists.** A transformer operates on a sequence of integer **token** ids, each mapped to a vector by the embedding table (NB11). So we need a rule to turn raw text into integers. Two naive options both fail:
- **Character-level** (what NB11 used): tiny vocab, but sequences are *very long* (attention is $O(T^2)$!) and each token carries little meaning.
- **Word-level**: short sequences, but the vocabulary is huge, and any word not seen in training is a dreaded out-of-vocabulary `<UNK>`.

**BPE is the sweet spot:** start from bytes/characters and *learn* a vocabulary of the most useful sub-word chunks by repeatedly merging the most frequent adjacent pair. Common words become single tokens; rare words break into a few sub-word pieces; nothing is ever out-of-vocabulary (worst case, it falls back to bytes). This is the algorithm behind GPT-2/3/4's tokenizers.


## 0. The BPE algorithm

Three primitives:
1. **`get_stats`** — count how often each adjacent pair of tokens occurs.
2. **`merge`** — replace every occurrence of a chosen pair with a new single token id.
3. **train** — repeat: find the most frequent pair, mint a new token for it, record the merge. Do this `num_merges` times to grow the vocabulary.

We work at the **byte level** (like GPT-2): the base vocabulary is the 256 possible bytes, so *any* text (any language, emoji, code) is representable — no `<UNK>` ever. Merges then build larger tokens on top.


In [1]:
from collections import Counter

def get_stats(ids):
    """Count occurrences of each adjacent pair."""
    return Counter(zip(ids, ids[1:]))

def merge(ids, pair, new_id):
    """Replace every occurrence of `pair` with `new_id`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids)-1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

# tiny demo of one merge
demo = [1, 2, 3, 1, 2, 4, 1, 2]
print("before:", demo)
print("most frequent pair:", get_stats(demo).most_common(1))
print("after merging (1,2)->99:", merge(demo, (1,2), 99))


before: [1, 2, 3, 1, 2, 4, 1, 2]
most frequent pair: [((1, 2), 3)]
after merging (1,2)->99: [99, 3, 99, 4, 99]


In [2]:
class BPETokenizer:
    def __init__(self):
        self.merges = {}          # (int,int) -> new_id, in the order learned
        self.vocab = {i: bytes([i]) for i in range(256)}   # id -> bytes

    def train(self, text, vocab_size, verbose_examples=0):
        ids = list(text.encode("utf-8"))          # start from raw bytes
        n_merges = vocab_size - 256
        shown = 0
        for m in range(n_merges):
            stats = get_stats(ids)
            if not stats: break
            pair = max(stats, key=stats.get)      # most frequent adjacent pair
            new_id = 256 + m
            ids = merge(ids, pair, new_id)
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if shown < verbose_examples:
                print(f"  merge {m:3d}: {pair} -> {new_id}  ({self.vocab[new_id]!r})")
                shown += 1
        return ids

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            # merge the pair that was learned EARLIEST (lowest new_id) among present pairs
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break     # nothing left to merge
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

print("BPETokenizer defined")


BPETokenizer defined


## 1. Train it and watch merges form real sub-words

We train on a chunk of text and print the first few merges. Watch how BPE discovers meaningful units — common letter pairs, then whole common words like `the` and `and` — purely from frequency.


In [3]:
import urllib.request
try:
    corpus = urllib.request.urlopen(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        timeout=15).read().decode("utf-8")[:200000]
    print("trained on tiny-shakespeare chunk,", len(corpus), "chars")
except Exception as e:
    print("offline, using bundled text:", repr(e)[:50])
    corpus = ("the quick brown fox jumps over the lazy dog. "
              "the theory of the thing is that the there and their differ. and and and the the the ") * 2000

tok = BPETokenizer()
print("first merges learned:")
tok.train(corpus, vocab_size=512, verbose_examples=12)
print(f"\nvocabulary size: {len(tok.vocab)} ({len(tok.merges)} learned merges + 256 base bytes)")


trained on tiny-shakespeare chunk, 200000 chars
first merges learned:
  merge   0: (101, 32) -> 256  (b'e ')
  merge   1: (116, 104) -> 257  (b'th')
  merge   2: (116, 32) -> 258  (b't ')
  merge   3: (115, 32) -> 259  (b's ')


  merge   4: (111, 117) -> 260  (b'ou')
  merge   5: (100, 32) -> 261  (b'd ')
  merge   6: (44, 32) -> 262  (b', ')
  merge   7: (101, 114) -> 263  (b'er')
  merge   8: (97, 110) -> 264  (b'an')
  merge   9: (105, 110) -> 265  (b'in')


  merge  10: (121, 32) -> 266  (b'y ')
  merge  11: (58, 10) -> 267  (b':\n')



vocabulary size: 512 (256 learned merges + 256 base bytes)


**What to notice.** The earliest merges are the most frequent pairs — letter combinations like `t`+`h`→`th`, then `th`+`e`→`the`, building up whole common words. BPE *discovers* that "the" deserves its own token because it appears so often, with no linguistic knowledge — pure data-driven frequency. Rare words never get merged into single tokens, so they stay as multiple sub-word pieces. That's how a ~50k-token vocabulary covers all of English (and beyond) without any out-of-vocabulary word.


## 2. Round-trip correctness and compression

Two things every tokenizer must do: **decode(encode(x)) == x** exactly (lossless), and **compress** — represent text in fewer tokens than characters (shorter sequences = cheaper attention). Let's verify both.


In [4]:
sample = "the theory of the thing"
ids = tok.encode(sample)
print(f"text     : {sample!r}  ({len(sample)} chars)")
print(f"tokens   : {ids}  ({len(ids)} tokens)")
print(f"decoded  : {tok.decode(ids)!r}")
assert tok.decode(ids) == sample, "round-trip failed!"
print("✅ round-trip lossless")

# compression on a fresh held-out sentence
test = "the quick brown fox jumps over the lazy dog many times"
n_chars = len(test); n_tokens = len(tok.encode(test))
print(f"\ncompression: {n_chars} chars -> {n_tokens} tokens  ({n_chars/n_tokens:.2f} chars/token)")


text     : 'the theory of the thing'  (23 chars)
tokens   : [307, 336, 270, 266, 289, 293, 257, 295]  (8 tokens)
decoded  : 'the theory of the thing'
✅ round-trip lossless

compression: 54 chars -> 33 tokens  (1.64 chars/token)


**What to notice.** Encoding then decoding returns the *exact* original text — BPE is lossless (crucial: the model's output must map back to real text). And we compress several characters into each token (~3–4 chars/token is typical for English; GPT-4's tokenizer averages ~4). Fewer tokens per sentence means shorter sequences, so cheaper $O(T^2)$ attention and more text fitting in a fixed context window — tokenization efficiency directly affects cost and context length.


## 3. Compare to `tiktoken` (GPT's real tokenizer)

Our from-scratch BPE is the same algorithm OpenAI uses. Let's compare against `tiktoken`'s `cl100k_base` (the GPT-4 tokenizer) to see real-world sub-word tokenization.


In [5]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")     # GPT-4's tokenizer
for s in ["Hello world!", "tokenization", "antidisestablishmentarianism", "  spaces   matter"]:
    toks = enc.encode(s)
    pieces = [enc.decode([t]) for t in toks]
    print(f"{s!r:36s} -> {len(toks)} tokens: {pieces}")
print(f"\ncl100k_base vocab size: {enc.n_vocab:,}")


'Hello world!'                       -> 3 tokens: ['Hello', ' world', '!']
'tokenization'                       -> 2 tokens: ['token', 'ization']
'antidisestablishmentarianism'       -> 6 tokens: ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']
'  spaces   matter'                  -> 4 tokens: [' ', ' spaces', '  ', ' matter']

cl100k_base vocab size: 100,277


**What to notice.** Real GPT tokens are sub-word pieces: common words are one token (`Hello`), long rare words split into morphemes (`antidis`, `establishment`, `arian`, `ism`), and **spaces attach to the following word** (` world` is one token including the leading space). That leading-space detail matters a lot — `"hello"` and `" hello"` are *different tokens*, which is why prompt formatting (trailing spaces!) can subtly change model behavior. Our from-scratch tokenizer does the same thing; `tiktoken` just has a 100k vocab trained on far more data.


## 4. Why tokenization causes LLMs' weirdest failures

Many notorious LLM quirks are *tokenization artifacts*, not reasoning failures. Let's demonstrate three concretely.


In [6]:
print("=== (a) Numbers tokenize inconsistently -> bad arithmetic ===")
for num in ["7", "42", "127", "1000", "1234567"]:
    toks = enc.encode(num)
    print(f"  {num:>8s} -> {len(toks)} token(s): {[enc.decode([t]) for t in toks]}")
print("  A number like 1234567 splits into arbitrary chunks, so the model never sees clean digits.")

print("\n=== (b) 'Count the letters' is hard because the model sees TOKENS, not letters ===")
word = "strawberry"
toks = enc.encode(word)
print(f"  {word!r} -> tokens {[enc.decode([t]) for t in toks]}")
print(f"  The model sees {len(toks)} chunks, not 10 letters — so counting the r's is unnatural for it.")

print("\n=== (c) Non-English text is far less efficient (fewer merges learned for it) ===")
for label, s in [("English", "hello how are you today"), ("Chinese", "你好吗今天怎么样"),
                 ("emoji", "\U0001f600\U0001f680\U0001f9e0\U0001f525")]:
    print(f"  {label:8s}: {len(s)} chars -> {len(enc.encode(s))} tokens ({len(enc.encode(s))/max(len(s),1):.2f} tok/char)")


=== (a) Numbers tokenize inconsistently -> bad arithmetic ===
         7 -> 1 token(s): ['7']
        42 -> 1 token(s): ['42']
       127 -> 1 token(s): ['127']
      1000 -> 2 token(s): ['100', '0']
   1234567 -> 3 token(s): ['123', '456', '7']
  A number like 1234567 splits into arbitrary chunks, so the model never sees clean digits.

=== (b) 'Count the letters' is hard because the model sees TOKENS, not letters ===
  'strawberry' -> tokens ['str', 'aw', 'berry']
  The model sees 3 chunks, not 10 letters — so counting the r's is unnatural for it.

=== (c) Non-English text is far less efficient (fewer merges learned for it) ===
  English : 23 chars -> 5 tokens (0.22 tok/char)
  Chinese : 8 chars -> 10 tokens (1.25 tok/char)
  emoji   : 4 chars -> 11 tokens (2.75 tok/char)


**What to notice.** These aren't reasoning bugs — they're *tokenization* bugs:
- **(a) Arithmetic:** `1234567` shatters into arbitrary multi-digit chunks that differ from how `127` or `1000` split, so the model can't line up place values — a big reason LLMs fumble long multiplication (newer tokenizers split digits individually to help).
- **(b) "How many r's in strawberry?":** the model literally never sees the letters — it sees a few sub-word tokens. Counting characters inside a token is like asking you to count the pixels in a word you're reading. (This is *the* famous example.)
- **(c) Non-English cost:** the tokenizer learned mostly English merges, so Chinese/emoji get far more tokens per character — meaning non-English users pay more and get less context. A real fairness/efficiency issue.

**The lesson for the residency:** when an LLM fails at something "simple," ask whether the *tokenizer* mangled the input before the model ever saw it. Tokenization is a leaky, consequential abstraction sitting between text and the model.

## 5. What you built / learned
- **BPE from scratch**: `get_stats` → `merge` → iterative training; byte-level base so nothing is OOV.
- **Lossless round-trip** and **compression** (chars → fewer tokens).
- Compared to **`tiktoken`** (GPT-4's real tokenizer); saw sub-word pieces and space-attachment.
- Traced three real LLM failure modes (**arithmetic, letter-counting, non-English cost**) directly to tokenization.

## 6. Exercises
1. **Vocab-size sweep.** Train BPE with vocab 300, 1000, 5000; plot chars/token (compression) vs vocab size. Where are diminishing returns?
2. **Digit tokenization.** Force each digit to be its own token (pre-split numbers before BPE). Does a small model then do addition better?
3. **Regex pre-tokenization.** GPT-2 splits text with a regex before BPE (so merges don't cross word/punctuation boundaries). Implement it and compare token quality.
4. **Multilingual fairness.** Train BPE on a bilingual corpus; measure tok/char for each language vs an English-only tokenizer.
5. **Reverse a string.** Show a model (or reason about why) reversing "strawberry" is hard token-wise; propose a fix.
6. **Special tokens.** Add `<|endoftext|>` and other special tokens; how are they handled so they never split?

## 7. Interview / residency framing
- *"Why not just use characters or words?"* → chars → long sequences ($O(T^2)$ attention) + weak tokens; words → huge vocab + OOV. BPE is the sub-word compromise.
- *"How does BPE work?"* → start from bytes, repeatedly merge the most frequent adjacent pair into a new token; common words become single tokens, rare words split; byte fallback ⇒ no OOV.
- *"Why does the LLM fail at arithmetic / counting letters?"* → tokenization: numbers split inconsistently; the model sees sub-word tokens, not individual characters.
- *"Why do trailing spaces change outputs?"* → spaces attach to the following token, so `"hello"` ≠ `" hello"` as tokens.
- *"Why is non-English more expensive?"* → tokenizers learn mostly English merges, so other scripts use more tokens per character.

---
**Next:** *Tier 2 · Notebook 13 — Sampling & decoding*: greedy, temperature, top-k, top-p (nucleus), beam search, and speculative decoding — how the logits become actual generated text.


## 📚 References & papers

This notebook reproduces / builds on:

- **Neural Machine Translation of Rare Words with Subword Units (BPE)** — Sennrich, Haddow & Birch (2016), arXiv:1508.07909.
- **Language Models are Unsupervised Multitask Learners (GPT-2 byte-level BPE)** — Radford et al. (2019).
- **tiktoken** — OpenAI. GPT-3.5/4 tokenizer (cl100k_base) used for comparison.